# 02 — Historical fire mapping
## Fire history around Galičica using EFFIS + Earth Engine

In this practical we use the **EFFIS burned-area polygon archive** to reconstruct recent fire history around Galičica, then use a consistent Sentinel-2 NBR time series to inspect the 2024 disturbance and early recovery.

By the end you will have:
- a fire-history table and map;
- annual fire counts and mapped area;
- a focused view of the August 2024 event;
- an EO time series over the main 2024 Ohrid-side burned polygon;
- a short interpretation for the Galičica capstone.

> EFFIS polygons are a reference fire-history product, not field truth. Separate polygons can represent different parts of the same broader fire episode.

## 1. Imports and Earth Engine

In [ ]:
from pathlib import Path
import ee
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium

GEE_PROJECT_ID = "ee-andreydara"

try:
    ee.Initialize(project=GEE_PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)

print("Earth Engine ready.")

## 2. Locate the EFFIS teaching layer

The small Galičica subset is about 0.3 MB and contains the useful EFFIS attributes.  
Put it in:

`~/mystorage/fire-school/data/effis/Galicica.gpkg`

The code also accepts the earlier filename `galicica_effis.gpkg`.

In [ ]:
repo_root = Path.home() / "mystorage" / "fire-school"

candidates = [
    repo_root / "data" / "effis" / "Galicica.gpkg",
    repo_root / "data" / "effis" / "galicica_effis.gpkg",
    Path.cwd().parent / "data" / "effis" / "Galicica.gpkg",
    Path.cwd().parent / "data" / "effis" / "galicica_effis.gpkg",
]

EFFIS_PATH = next((p for p in candidates if p.exists()), None)

if EFFIS_PATH is None:
    raise FileNotFoundError(
        "Galičica EFFIS GeoPackage not found. "
        "Place Galicica.gpkg in fire-school/data/effis/."
    )

print("Using:", EFFIS_PATH)

## 3. Read and inspect the fire archive

In [ ]:
effis = gpd.read_file(EFFIS_PATH).to_crs("EPSG:4326").copy()

effis["FIREDATE"] = pd.to_datetime(effis["FIREDATE"], errors="coerce")
effis["FINALDATE"] = pd.to_datetime(effis["FINALDATE"], errors="coerce")
effis["AREA_HA"] = pd.to_numeric(effis["AREA_HA"], errors="coerce")
effis["year"] = effis["FIREDATE"].dt.year

print("Features:", len(effis))
print("Years:", int(effis["year"].min()), "–", int(effis["year"].max()))
print("Total mapped area (sum of EFFIS polygon attributes):",
      round(effis["AREA_HA"].sum(), 0), "ha")

display(
    effis[
        ["id", "FIREDATE", "FINALDATE", "COUNTRY", "COMMUNE", "AREA_HA", "CLASS"]
    ]
    .sort_values("FIREDATE")
    .tail(10)
)

### Question

Before plotting anything: do you expect **number of fires** and **total burned area** to tell the same story? Why not?

## 4. Fire history by year

In [ ]:
annual = (
    effis.groupby("year", dropna=True)
    .agg(
        n_polygons=("id", "count"),
        mapped_area_ha=("AREA_HA", "sum"),
    )
    .reset_index()
)

annual

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(annual["year"].astype(str), annual["mapped_area_ha"])
ax.set_ylabel("EFFIS mapped area (ha)")
ax.set_xlabel("Year")
ax.set_title("Mapped burned area in the Galičica-area EFFIS subset")
plt.xticks(rotation=45)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(annual["year"].astype(str), annual["n_polygons"])
ax.set_ylabel("Number of EFFIS polygons")
ax.set_xlabel("Year")
ax.set_title("EFFIS fire polygons by year")
plt.xticks(rotation=45)
plt.show()

## 5. Map the historical polygons

In [ ]:
CENTER = [41.02, 20.93]
ZOOM = 10

# Folium/GeoJSON cannot serialize pandas Timestamp objects directly.
effis_map = effis.copy()
for col in ["FIREDATE", "FINALDATE"]:
    effis_map[col] = effis_map[col].dt.strftime("%Y-%m-%d")

m = folium.Map(location=CENTER, zoom_start=ZOOM, tiles="CartoDB positron")

folium.GeoJson(
    effis_map,
    name="EFFIS fire history",
    tooltip=folium.GeoJsonTooltip(
        fields=["id", "FIREDATE", "COUNTRY", "COMMUNE", "AREA_HA"],
        aliases=["ID", "Start", "Country", "Commune", "Area (ha)"],
        localize=True,
    ),
    style_function=lambda _: {
        "color": "#b22222",
        "weight": 1,
        "fillOpacity": 0.15,
    },
).add_to(m)

folium.LayerControl().add_to(m)
m

### Interpretation

Look for:
- repeated fire occurrence in the same broad landscape;
- very large versus very small events;
- administrative splitting of nearby/cross-border events;
- places with apparently little mapped fire history.

Absence of an EFFIS polygon does **not** prove absence of fire.

## 6. Focus on August 2024

The EFFIS subset contains several polygons around the broader August 2024 episode.  
For the North Macedonia / Ohrid side, the large polygon with EFFIS ID **240575** is a useful reference for the next steps.

In [ ]:
aug2024 = effis[
    (effis["FIREDATE"] >= "2024-08-01") &
    (effis["FIREDATE"] <= "2024-08-18")
].copy()

display(
    aug2024[
        ["id", "FIREDATE", "FINALDATE", "COUNTRY", "COMMUNE", "AREA_HA"]
    ].sort_values(["FIREDATE", "AREA_HA"], ascending=[True, False])
)

target = effis[effis["id"].astype(str) == "240575"].copy()

if target.empty:
    raise RuntimeError("Expected EFFIS reference polygon 240575 was not found.")

print("Target reference area:", float(target.iloc[0]["AREA_HA"]), "ha")

In [ ]:
aug2024_map = aug2024.copy()
target_map = target.copy()

for frame in [aug2024_map, target_map]:
    for col in ["FIREDATE", "FINALDATE"]:
        frame[col] = frame[col].dt.strftime("%Y-%m-%d")

m2024 = folium.Map(location=[40.93, 20.84], zoom_start=11, tiles="CartoDB positron")

folium.GeoJson(
    aug2024_map,
    name="August 2024 EFFIS polygons",
    tooltip=folium.GeoJsonTooltip(
        fields=["id", "FIREDATE", "COUNTRY", "COMMUNE", "AREA_HA"],
        aliases=["ID", "Start", "Country", "Commune", "Area (ha)"],
    ),
    style_function=lambda f: {
        "color": "black",
        "weight": 2,
        "fillOpacity": 0.12,
    },
).add_to(m2024)

folium.GeoJson(
    target_map,
    name="Target: EFFIS 240575",
    style_function=lambda _: {
        "color": "#d73027",
        "weight": 3,
        "fillOpacity": 0.20,
    },
).add_to(m2024)

folium.LayerControl().add_to(m2024)
m2024

## 7. Build a consistent EO time series over the target polygon

To compare years fairly, we use the **same seasonal window every year**:  
**20 August – 15 October**.

We calculate median Sentinel-2 NBR and then the mean NBR inside the EFFIS target polygon.

This is not a full fire-detection algorithm. It is a compact way to see whether the known 2024 disturbance is visible in the spectral trajectory.

In [ ]:
def mask_s2_scl(img):
    scl = img.select("SCL")
    bad = (
        scl.eq(3)
        .Or(scl.eq(8))
        .Or(scl.eq(9))
        .Or(scl.eq(10))
        .Or(scl.eq(11))
    )
    return img.updateMask(bad.Not())

target_geom = target.geometry.iloc[0]
target_ee = ee.Geometry(target_geom.__geo_interface__)

def seasonal_nbr(year):
    start = f"{year}-08-20"
    end = f"{year}-10-16"

    col = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(target_ee)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
        .map(mask_s2_scl)
    )

    nbr = (
        col.median()
        .normalizedDifference(["B8", "B12"])
        .rename("NBR")
    )

    mean_nbr = nbr.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=target_ee,
        scale=20,
        maxPixels=1e8,
        bestEffort=True,
    ).get("NBR")

    return ee.Feature(
        None,
        {
            "year": year,
            "n_scenes": col.size(),
            "mean_nbr": mean_nbr,
        },
    )

years = list(range(2017, 2026))
series_fc = ee.FeatureCollection([seasonal_nbr(y) for y in years])

records = series_fc.getInfo()["features"]
nbr_ts = pd.DataFrame([f["properties"] for f in records]).sort_values("year")
nbr_ts

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(nbr_ts["year"], nbr_ts["mean_nbr"], marker="o")
ax.axvline(2024, linestyle="--", alpha=0.7)
ax.set_xlabel("Year")
ax.set_ylabel("Mean late-summer NBR")
ax.set_title("Sentinel-2 NBR over EFFIS polygon 240575")
ax.grid(alpha=0.25)
plt.show()

### Questions

1. Is 2024 visibly different from the previous years?
2. What happens in 2025?
3. Could differences between years reflect something other than fire?
4. Why did we keep the seasonal window fixed?
5. What would change if we used a single image rather than a median composite?

## 8. Simple before/after metric

In [ ]:
ts = nbr_ts.set_index("year")

if 2023 in ts.index and 2024 in ts.index:
    drop_2024 = ts.loc[2023, "mean_nbr"] - ts.loc[2024, "mean_nbr"]
    print("2023 → 2024 NBR drop:", round(float(drop_2024), 3))

if 2024 in ts.index and 2025 in ts.index:
    recovery_2025 = ts.loc[2025, "mean_nbr"] - ts.loc[2024, "mean_nbr"]
    print("2024 → 2025 NBR increase:", round(float(recovery_2025), 3))

## 9. Stretch tasks

Choose one:

### A — Different seasonal window
Use July–August instead of late August–October. Does the trajectory change?

### B — Another EFFIS polygon
Select a different historical fire polygon and repeat the NBR time series.

### C — Repeated-fire question
Identify a place where historical EFFIS polygons overlap or lie very close together.  
What extra analysis would you need before calling this a **recurrently burned site**?

### D — Compare NDVI and NBR
Build the same annual trajectory using NDVI. Which index shows the 2024 disturbance more clearly?

## 10. Output for the Galičica capstone

Keep:
- the historical fire map;
- the annual burned-area chart;
- the 2024 reference map;
- the NBR trajectory;
- **three findings**;
- **one limitation**;
- **one management-relevant interpretation**.

The next practical, **03 — Burned area and burn severity**, maps the 2024 spectral change spatially with dNBR.